# 🔤 From Text to Attention

By the end of this notebook, you'll understand:
1. How text becomes tokens (and why it matters)
2. How tokens become meaningful vectors (embeddings)
3. How position information gets added
4. How attention actually works — coded from scratch

Let's start!

---
# Part 1: Tokenization
*What does the model actually see?*

---

In [2]:
# First, install tiktoken (OpenAI's tokenizer)
%pip install tiktoken -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import tiktoken
import numpy as np

In [4]:
tokenizer=tiktoken.get_encoding("o200k_base")
encode=tokenizer.encode("hello world")
print(encode)
assert tokenizer.decode(tokenizer.encode("hello world"))=="hello world"
# to get the tokeniser corresponding to a specific model in the openAI API:
enc=tiktoken.encoding_for_model("gpt-4o")
print(f"vocabulary size: {tokenizer.n_vocab:,}")

[24912, 2375]
vocabulary size: 200,019


## 1.1 Basic Tokenization

Let's see how text gets converted to numbers.

In [5]:
#simple example
text="hello world"
tokens=tokenizer.encode(text)
print(f"text:{text}")
print(f"Tokens:{tokens}")
print(f"number of tokens: {len(tokens)}")

#let's see what each token represents
print("\nToken Breakdown:")
for token in tokens:
    print(f"{token}->{tokenizer.decode([token])}")
    


text:hello world
Tokens:[24912, 2375]
number of tokens: 2

Token Breakdown:
24912->hello
2375-> world


Notice: "Hello" is one token, but " world" (with the space!) is another.
The space attaches to the following word.

In [6]:
#let's try more interesting examples
examples = [
    "Hello world",
    "don't",
    "artificial intelligence",
    "I love AI",
    "supercalifragilisticexpialidocious",
    "🚀",
    "café",
    "    spaces    ",  # multiple spaces
]
print("how different texts get tokenized:\n")
for text in examples:
    tokens=tokenizer.encode(text)
    print(f"{text}")
    print(f"{len(tokens)} tokens:{tokens}")
    # show the pieces
    piece=[]
    for t in tokens:
        pieces=tokenizer.decode([t])
        piece.append(pieces)
    print(f"pieces:{piece}")
    print()

how different texts get tokenized:

Hello world
2 tokens:[13225, 2375]
pieces:['Hello', ' world']

don't
1 tokens:[91418]
pieces:["don't"]

artificial intelligence
3 tokens:[497, 20454, 22990]
pieces:['art', 'ificial', ' intelligence']

I love AI
3 tokens:[40, 3047, 20837]
pieces:['I', ' love', ' AI']

supercalifragilisticexpialidocious
10 tokens:[17789, 5842, 366, 17764, 311, 6207, 8067, 563, 315, 170661]
pieces:['super', 'cal', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']

🚀
2 tokens:[112927, 222]
pieces:['�', '�']

café
2 tokens:[66, 103112]
pieces:['c', 'afé']

    spaces    
3 tokens:[271, 18608, 257]
pieces:['   ', ' spaces', '    ']



## 🧪 Try It Yourself

Tokenize your own text! Try:
- Your name
- A sentence in another language
- Some code
- Emojis

In [7]:
my_text="my name is sidhhay"
tokens=tokenizer.encode(my_text)
print(f"my text ->{my_text}")
print(f"number of tokens ->{len(tokens)}")
print(f"encoded tokens:{tokens}")
pieces=[]
print(tokenizer.decode(tokens))
for t in tokens:
    #print([t])
    #print(t)
    pieces.append(tokenizer.decode([t]))

print(f"pieces:{pieces}")

my text ->my name is sidhhay
number of tokens ->6
encoded tokens:[3825, 1308, 382, 265, 10661, 39679]
my name is sidhhay
pieces:['my', ' name', ' is', ' s', 'idh', 'hay']


## 1.2 Why Tokenization Matters

Token count affects:
- API costs (you pay per token)
- Context limits (GPT-4 has 128K token limit)
- Model behavior (some tasks break across token boundaries)
- model has context limits
If your text tokenizes inefficiently:
 You hit limit faster
 Model forgets earlier parts
 Better tokenization = more usable context.

⚡  It Affects Speed

Tokenization happens before every inference.

Slow tokenizer = slower API response.

That’s why tiktoken is optimized.
# Tokenization = the language the model speaks internally

In [8]:
# Compare token efficiency across different content types
test_cases = {
    "English prose": "The quick brown fox jumps over the lazy dog.",
    "Python code": "def hello():\n    print('Hello, world!')",
    "JSON": '{"name": "Alice", "age": 30, "city": "NYC"}',
    "Numbers": "1234567890 9876543210 1111111111",
    "URL": "https://www.example.com/path/to/page?query=value",
}
print("token efficency comparison:\n")
for name,text in test_cases.items():
    tokens=tokenizer.encode(text)
    chars=len(text)
    ratio=chars/len(tokens)
    print(f"{name}:")
    print(f" {chars} chars → {len(tokens)} tokens ({ratio:.1f} chars/token)")


token efficency comparison:

English prose:
 44 chars → 10 tokens (4.4 chars/token)
Python code:
 39 chars → 11 tokens (3.5 chars/token)
JSON:
 43 chars → 19 tokens (2.3 chars/token)
Numbers:
 32 chars → 14 tokens (2.3 chars/token)
URL:
 48 chars → 11 tokens (4.4 chars/token)


**Key insight**: Different content has different token efficiency.
- English prose: ~4 characters per token
- Code/JSON: often less efficient (more tokens per character)
- This affects your API costs!

## 1.3 The Token Boundary Problem

Some tasks are hard because they require reasoning WITHIN tokens.

- It’s not a problem if you work at token level.

It becomes a problem when you:

- Estimate using characters
- Estimate using words
- Cut text blindly
- Filter strings without tokenizing

In [9]:
word="strawberry"
tokens=tokenizer.encode(word)
print(f"Word:'{word}'")
print(f"Tokens:{tokens}")
pieces=[]
for t in tokens:
    p=tokenizer.decode([t])
    pieces.append(p)
print(f"Pieces:{pieces}")
print()
print("The model sees these pieces, not individual letters!")
print("Counting 'r's requires looking INSIDE tokens — that's hard.")

Word:'strawberry'
Tokens:[302, 1618, 19772]
Pieces:['st', 'raw', 'berry']

The model sees these pieces, not individual letters!
Counting 'r's requires looking INSIDE tokens — that's hard.


In [10]:
#Another example:reversing words
word="hello"
tokens=tokenizer.encode(word)
print(f"'{word}' -> tokens:{[tokenizer.decode([t]) for t in tokens]}")
print()
print("If 'hello' is ONE token, the model can't easily reverse it.")
print("It would need to decompose something it sees as atomic.")

'hello' -> tokens:['hello']

If 'hello' is ONE token, the model can't easily reverse it.
It would need to decompose something it sees as atomic.


---
# ✅ Checkpoint: Tokenization Complete

now we understand:
- Text → Token IDs (integers)
- Subword tokenization (BPE)
- Why token counts matter
- Why some tasks are hard (token boundaries)

 lets dive to another bit that is embeddings 


---
# Part 2: Embeddings
*Tokens as points in space*

---

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
# we'll use a small pre-computed set of embeddings for demonstration

## 2.1 What's an Embedding?

An embedding converts a token ID into a vector of numbers.
Let's simulate this with random embeddings first, then use real ones.

In [12]:
#simulating an embedding matrix
vocab_size=50000
embedding_dim=768 # gpt-2 size
# Random embedding matrix (in reality, this is LEARNED)
embedding_matrix = np.random.randn(vocab_size,embedding_dim)*0.02

print(f"Embedding matrix shape: {embedding_matrix.shape}")
print(f"  → {vocab_size:,} tokens")
print(f"  → {embedding_dim} dimensions per token")
print(f"  → {vocab_size * embedding_dim:,} total parameters just for embeddings!")

Embedding matrix shape: (50000, 768)
  → 50,000 tokens
  → 768 dimensions per token
  → 38,400,000 total parameters just for embeddings!


In [13]:
# How embedding lookup works
text="hello world"
tokens=tokenizer.encode(text)
print(f"Text: '{text}'")
print(f"Token IDs: {tokens}")
print()
#Look Up token's embedding
for token_id in tokens:
    embedding=embedding_matrix[token_id]
    print(f"Token {token_id} ('{tokenizer.decode([token_id])}')")
    print(f"  → Embedding shape: {embedding.shape}")
    print(f"  → First 10 values: {embedding[:10].round(3)}")
    print()



Text: 'hello world'
Token IDs: [24912, 2375]

Token 24912 ('hello')
  → Embedding shape: (768,)
  → First 10 values: [ 0.004 -0.033 -0.002 -0.002 -0.024 -0.014  0.054 -0.039 -0.    -0.011]

Token 2375 (' world')
  → Embedding shape: (768,)
  → First 10 values: [-0.014  0.005 -0.016  0.033  0.002 -0.016 -0.    -0.027  0.008  0.007]



## 2.2 Real Embeddings: Word Similarity

Let's use pre-trained word vectors to see embeddings in action.
We'll use Gensim's Word2Vec — classic but illustrative.

In [14]:
%pip install gensim -q
#it helps you convert text into vectors and analyse meaning.

Note: you may need to restart the kernel to use updated packages.


In [15]:
import gensim.downloader as api
# load pre-trained word vectors 
word_vectors=api.load("glove-wiki-gigaword-100") #100-dimensional Glove
print(f"Loaded! Vocabulary: {len(word_vectors):,} words")

Loaded! Vocabulary: 400,000 words


🔹 2. Key Idea: “You shall know a word by the company it keeps”

In [ ]:
# Find similar words

word="king"
similar=word_vectors.most_similar(word,topn=10)
print(f"Words most similar to '{word}' :\n")
# for similar words they appear in similar context,so their vectors 
# become close in space
# if angle between vectors small they are similar and vice versa.
for similar_word,score in similar:
    print(f"{similar_word}:{score:.3f}")

Words most similar to 'king' :


prince:0.768
queen:0.751
son:0.702
brother:0.699
monarch:0.698
throne:0.692
kingdom:0.681
father:0.680
emperor:0.671
ii:0.668


# 2.3 -The famous analogy: king - man + woman = ?

In [22]:
#vector arthmetic with words!
result=word_vectors.most_similar(
    positive=["king","woman"],
    negative=["man"],
    topn=5
)
print("king-man+woman=?\n")
for word,score in result:
    print(f"{word}:{score:.3f}")


king-man+woman=?

queen:0.770
monarch:0.684
throne:0.676
daughter:0.659
princess:0.652


In [23]:
# lets see more analogies
analogies = [
    (["paris", "germany"], ["france"], "paris - france + germany = ?"),
    (["bigger", "cold"], ["big"], "bigger - big + cold = ?"),
]
for positive,negative,description in analogies:
    print(description)
    result=word_vectors.most_similar(positive=positive,negative=negative,topn=3)
    for word,score in result:
        print(f"{word}:{score:.3f}")
    print();    

paris - france + germany = ?
berlin:0.885
frankfurt:0.799
vienna:0.768

bigger - big + cold = ?
cooler:0.688
warmer:0.685
colder:0.675



## 2.4 Visualizing Embedding Space

Embeddings are high-dimensional. Let's project to 2D to visualize.

In [ ]:
# embeddings for a set of related words
word_groups = {
    "royalty": ["king", "queen", "prince", "princess", "royal", "throne"],
    "family": ["man", "woman", "boy", "girl", "father", "mother"],
    "animals": ["dog", "cat", "horse", "bird", "fish", "lion"],
    "tech": ["computer", "software", "internet", "digital", "data", "code"],
}
words